# What vLLM's tool-call parser does

Starts vLLM twice on the same T4 and sends the same request each time (`serving/raw_tool_response.py`: Nova's
system prompt, the six tools, "Where is my order 11904?", greedy), keeping the raw response JSON:

1. **With the parser** (`--enable-auto-tool-choice --tool-call-parser qwen3_coder`, the serving flags): the call
   should arrive parsed in `tool_calls`.
2. **Without the parser** (both flags removed; vLLM accepts no `--enable-auto-tool-choice` without a parser): does
   the request get refused, and where does the call end up?

Each server gets two requests: `tool_choice="auto"` (what the agent loop sends) and `tool_choice="none"` (tools still
in the prompt, server told not to parse).

**Before running (Kaggle):** GPU **T4 x2** (one is used), Internet **On**; secret `HF_TOKEN` optional (avoids download
rate limits). No database needed. **Save & Run All**; about 15–25 minutes, most of it installing vLLM and loading
the model twice.

**After:** Output → download **`tool-call-parser.zip`** and unzip it into the repo's `notes/tool-call-anatomy/`.

In [ ]:
# 1. Clone the repo and install the client side (the script imports the repo's tool schemas and prompt).
import os, subprocess, sys, time, urllib.request
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; the model is public, downloads may be slower")
REPO_DIR = os.path.abspath("customer-support-SLM")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/jagjotsingh244-source/customer-support-SLM.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
%pip install -q "psycopg[binary]" pgvector pydantic
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. vLLM in its own environment, pinned to the versions that passed the GPU risk tests.
!uv --version || pip install -q uv
!uv venv -q /tmp/vllm-env --python 3.12
!uv pip install -q --python /tmp/vllm-env/bin/python -r requirements-serve.txt

In [ ]:
# 3. Start / stop helpers. The libcuda fix (serving/t4_libcuda_fix.sh) is applied through LIBRARY_PATH.
fix = subprocess.run(["bash", "-c", "source serving/t4_libcuda_fix.sh >&2; echo $LIBRARY_PATH"],
                     capture_output=True, text=True)
print(fix.stderr.strip())
ENV = {**os.environ, "LIBRARY_PATH": fix.stdout.strip(), "CUDA_VISIBLE_DEVICES": "0"}
BASE = ["/tmp/vllm-env/bin/vllm", "serve", "Qwen/Qwen3.5-2B", "--dtype", "half", "--max-model-len", "8192",
        "--language-model-only", "--port", "8000"]
PARSER = ["--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder"]
OUT = "/kaggle/working/tool-call-parser"
os.makedirs(OUT, exist_ok=True)

def serve(label, flags):
    log = open(f"{OUT}/vllm_{label}.log", "w")
    proc = subprocess.Popen(BASE + flags, stdout=log, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < 900:
        if proc.poll() is not None:
            raise RuntimeError(f"vLLM exited; see {OUT}/vllm_{label}.log")
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=5)
            print(f"{label}: up after {time.time() - t0:.0f}s")
            return proc
        except Exception:
            time.sleep(5)
    raise RuntimeError("vLLM did not start in 15 minutes")

def stop(proc):
    os.killpg(proc.pid, 15)
    proc.wait(120)
    time.sleep(10)          # let the GPU memory go before the next start

In [ ]:
# 4. With the parser: the serving flags.
server = serve("with_parser", PARSER)
!python serving/raw_tool_response.py --label with-parser --out {OUT}/vllm_with_parser.json
stop(server)

In [ ]:
# 5. Without the parser: same server, both tool flags removed.
server = serve("without_parser", [])
!python serving/raw_tool_response.py --label without-parser --out {OUT}/vllm_without_parser.json
stop(server)

In [ ]:
# 6. Bundle the results and the server logs for the laptop.
!cd /kaggle/working && zip -qr tool-call-parser.zip tool-call-parser && ls -la tool-call-parser